In [15]:
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio import features
import geopandas as gpd
import numpy as np
import os

In [18]:
def align_raster_to_target(source_path, target_shape, target_transform, target_crs):
    """Reads and aligns a source raster to a target grid."""
    with rasterio.open(source_path) as src:
        aligned_data = np.zeros(target_shape, dtype=src.dtypes[0])
        reproject(
            source=rasterio.band(src, 1),
            destination=aligned_data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=target_transform,
            dst_crs=target_crs,
            resampling=Resampling.nearest
        )
        return aligned_data

def generate_flood_map_vector_lc(reference_path, event_path, vector_lc_path, output_path, nodata_value=255):
    """
    Generates a flood map using reference water, event water, and a vector land mask.
    
    Conditions for flooding:
    1. Event imagery shows water (1)
    2. Reference imagery shows land (0)
    3. Pixel falls inside a vector feature from the shapefile
    """
    
    print(f"Loading event mask (target grid): {event_path}")
    with rasterio.open(event_path) as event_src:
        event_data = event_src.read(1)
        event_meta = event_src.meta.copy()
        event_transform = event_src.transform
        event_crs = event_src.crs
        event_shape = event_data.shape
        
    print(f"Loading and aligning reference raster mask: {reference_path}")
    ref_aligned = align_raster_to_target(reference_path, event_shape, event_transform, event_crs)
    
    print(f"Loading and rasterizing vector land mask: {vector_lc_path}")
    # Load the shapefile
    gdf = gpd.read_file(vector_lc_path)
    
    # Reproject the shapefile to match the Event raster's CRS if they differ
    if gdf.crs != event_crs:
        print(f"Reprojecting vector from {gdf.crs} to {event_crs}...")
        gdf = gdf.to_crs(event_crs)
    
    # Rasterize the geometries into a boolean mask (1 = inside feature, 0 = outside)
    if len(gdf) > 0:
        lc_mask = features.rasterize(
            shapes=gdf.geometry,
            out_shape=event_shape,
            transform=event_transform,
            fill=0,          # Background value (not land)
            default_value=1, # Value assigned to areas inside polygons (land)
            dtype=np.uint8
        )
    else:
        print("Warning: Vector file is empty. No land detected.")
        lc_mask = np.zeros(event_shape, dtype=np.uint8)

    print("Calculating flood extent...")
    
    # Define boolean conditions
    is_event_water = (event_data == 1)
    is_ref_land = (ref_aligned == 0)
    is_vector_land = (lc_mask == 1)
    
    # A pixel is flooded ONLY if all three conditions are True
    flood_map = np.where(is_event_water & is_ref_land & is_vector_land, 1, 0).astype(np.uint8)
    
    # Apply NoData masking if either of the raster inputs contained the NoData value
    any_nodata = (event_data == nodata_value) | (ref_aligned == nodata_value)
    flood_map = np.where(any_nodata, nodata_value, flood_map)

    # Update metadata for output
    event_meta.update({
        'dtype': 'uint8',
        'count': 1,
        'nodata': nodata_value,
        'compress': 'lzw'
    })

    print(f"Saving flood map to: {output_path}")
    with rasterio.open(output_path, 'w', **event_meta) as dest:
        dest.write(flood_map, 1)
        
    print("Flood map generation complete!")

In [20]:
# Tafuna
REF_MASK = "/Users/sahuang/Documents/DATA/umbra/AS_Water_Masks/Tafuna/20250524.sub.wmask.tif"
EVENT_MASK = "/Users/sahuang/Documents/DATA/umbra/AS_Water_Masks/Tafuna/20250330.sub.wmask.tif"
# LANDCOVER_MASK = '/Users/sahuang/Documents/DATA/am_samoa/Landcover/am_samoa_ccap_ocean_mask_small.tif'
VECTOR_LC = "/Users/sahuang/Documents/DATA/am_samoa/Landcover/tafuna_ccap_ocean_mask_crop.shp"  # Can be .shp, .gpkg, .geojson, etc.
OUTPUT_MAP = "/Users/sahuang/Documents/DATA/umbra/AS_Water_Masks/Tafuna/Tafuna_flood_map.tif"

if all(os.path.exists(p) for p in [REF_MASK, EVENT_MASK, VECTOR_LC]):
    generate_flood_map_vector_lc(REF_MASK, EVENT_MASK, VECTOR_LC, OUTPUT_MAP)
else:
    print("Please update the script with valid file paths.")

Loading event mask (target grid): /Users/sahuang/Documents/DATA/umbra/AS_Water_Masks/Tafuna/20250330.sub.wmask.tif
Loading and aligning reference raster mask: /Users/sahuang/Documents/DATA/umbra/AS_Water_Masks/Tafuna/20250524.sub.wmask.tif
Loading and rasterizing vector land mask: /Users/sahuang/Documents/DATA/am_samoa/Landcover/tafuna_ccap_ocean_mask_crop.shp
Reprojecting vector from EPSG:32702 to EPSG:4326...
Calculating flood extent...
Saving flood map to: /Users/sahuang/Documents/DATA/umbra/AS_Water_Masks/Tafuna/Tafuna_flood_map.tif
Flood map generation complete!


In [22]:
# Pago Pago
REF_MASK = "/Users/sahuang/Documents/DATA/umbra/AS_Water_Masks/Pago/20250705.wmask.tif"
EVENT_MASK = "/Users/sahuang/Documents/DATA/umbra/AS_Water_Masks/Pago/20250330.wmask.tif"
VECTOR_LC = "/Users/sahuang/Documents/DATA/am_samoa/AmericanSamoa_3DEP_DEM_10m/pago/USGS_13_s14w171_20130911_lt_4m.shp"
OUTPUT_MAP = "/Users/sahuang/Documents/DATA/umbra/AS_Water_Masks/Pago/Pago_flood_map.tif"

if all(os.path.exists(p) for p in [REF_MASK, EVENT_MASK, VECTOR_LC]):
    generate_flood_map_vector_lc(REF_MASK, EVENT_MASK, VECTOR_LC, OUTPUT_MAP)
else:
    print("Please update the script with valid file paths.")

Loading event mask (target grid): /Users/sahuang/Documents/DATA/umbra/AS_Water_Masks/Pago/20250330.wmask.tif
Loading and aligning reference raster mask: /Users/sahuang/Documents/DATA/umbra/AS_Water_Masks/Pago/20250705.wmask.tif
Loading and rasterizing vector land mask: /Users/sahuang/Documents/DATA/am_samoa/AmericanSamoa_3DEP_DEM_10m/pago/USGS_13_s14w171_20130911_lt_4m.shp
Reprojecting vector from GEOGCS["GCS_NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433],AXIS["Longitude",EAST],AXIS["Latitude",NORTH]] to EPSG:4326...
Calculating flood extent...
Saving flood map to: /Users/sahuang/Documents/DATA/umbra/AS_Water_Masks/Pago/Pago_flood_map.tif
Flood map generation complete!
